# Project CV2: Robust Head Pose Estimation Under Facial Occlusions

Work by Antonio Vila Leis and Enric Baixauli Casañ

## Introduction

This project addresses the challenge of accurately estimating head posture in the presence of facial occlusions, such as masks and sunglasses.

TODO explicar todo

In [17]:
import os
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader
import sys
from dataset import LP300W, I2HeadDataset
import utils
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
import time
from models.utils_hpe import compute_euler_angles_from_rotation_matrices
import cv2
import torch._dynamo
torch._dynamo.config.suppress_errors = True

In [18]:
utils.set_global_seed(123)

if torch.cuda.is_available():
    device_name = "cuda"
elif torch.backends.mps.is_available():
    device_name = "mps"
else:
    device_name = "cpu"
    
device = torch.device(device_name)
print(f"Code runs in {device}")

Code runs in cuda


## 1. Data Loading

In [19]:
import os
import random
from torch.utils.data import DataLoader
import torchvision.transforms as T

RAW_DATASET_DIR = "./dataset/test/raw"
BATCH_SIZE = 32

transforms_pipeline = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

subjects_dict = {}

# Parse baseline references from the raw directory
for file in os.listdir(RAW_DATASET_DIR):
    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
        filename_no_ext = os.path.splitext(file)[0]
        
        # Extract subject prefix (e.g., pulls 'p01' from 'p01_image_01_raw' or 'p01_image_01')
        subject_id = filename_no_ext.split("_")[0] 
        
        full_path = os.path.join(RAW_DATASET_DIR, file)
        
        if subject_id not in subjects_dict:
            subjects_dict[subject_id] = []
        subjects_dict[subject_id].append(full_path)

unique_subjects = sorted(list(subjects_dict.keys()))
print(f"Total number of people found: {len(unique_subjects)}")

# Subject-level Split (10 Train / 1 Val / 1 Test)
random.seed(123)
random.shuffle(unique_subjects)

train_subj = unique_subjects[:9]
val_subj   = [unique_subjects[9]]
test_subj  = unique_subjects[10:]

print(f"Split people -> Train: {train_subj} | Val: {val_subj} | Test: {test_subj}")

# Gather lists of paths
train_paths = []
for subj in train_subj:
    train_paths.extend(subjects_dict[subj])

val_paths = []
for subj in val_subj:
    val_paths.extend(subjects_dict[subj])

test_paths = []
for subj in test_subj:
    test_paths.extend(subjects_dict[subj])

print(f"Split images -> Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")

# Instantiate Datasets
train_dataset = I2HeadDataset(image_paths_or_dir=train_paths, transform=transforms_pipeline, occlusion_mode="random")
val_dataset   = I2HeadDataset(image_paths_or_dir=val_paths, transform=transforms_pipeline, occlusion_mode="random")

# Define the 4 isolated evaluation tracks matching the test subject
test_raw     = I2HeadDataset(image_paths_or_dir=test_paths, transform=transforms_pipeline, occlusion_mode="raw")
test_mask    = I2HeadDataset(image_paths_or_dir=test_paths, transform=transforms_pipeline, occlusion_mode="mask")
test_glasses = I2HeadDataset(image_paths_or_dir=test_paths, transform=transforms_pipeline, occlusion_mode="glasses")
test_both    = I2HeadDataset(image_paths_or_dir=test_paths, transform=transforms_pipeline, occlusion_mode="both")

# Construct DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

raw_loader     = DataLoader(test_raw, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
mask_loader    = DataLoader(test_mask, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
glasses_loader = DataLoader(test_glasses, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
both_loader    = DataLoader(test_both, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print("DataLoaders built successfully!")

Total number of people found: 12
Split people -> Train: ['p08', 'p06', 'p10', 'p03', 'p04', 'p09', 'p12', 'p11', 'p07'] | Val: ['p02'] | Test: ['p05', 'p01']
Split images -> Train: 738 | Val: 82 | Test: 164
DataLoaders built successfully!


In [4]:
sys.path.append("./models")

from token_hpe import TokenHPE

model = TokenHPE(depth=3)
checkpoint = torch.load("./weights/TokenHPEv1-ViTB-224_224-lyr3.tar", map_location="cpu")

state_dict = checkpoint["model_state_dict"]

model.load_state_dict(state_dict, strict=True)

model = model.to(device)

==> Add Sine PositionEmbedding~


C:\Users\enric\AppData\Local\Temp\ipykernel_9028\1715322118.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("./weights/TokenHPEv1-ViTB-224_224-ly

In [5]:
model.eval()

print("\n-> Test 1/4: RAW")
mae_r, rmse_r, r2_r = utils.evaluate_metrics(model, raw_loader, device)

print("\n-> Test 2/4: MASK")
mae_m, rmse_m, r2_m = utils.evaluate_metrics(model, mask_loader, device)

print("\n-> Test 3/4: GLASSES")
mae_g, rmse_g, r2_g = utils.evaluate_metrics(model, glasses_loader, device)

print("\n-> Test 4/4: BOTH")
mae_b, rmse_b, r2_b = utils.evaluate_metrics(model, both_loader, device)

def print_results(test_name, mae, rmse, r2):
    print(f"\n{test_name}")
    print(f"  MAE -> Yaw: {mae['yaw']} | Pitch: {mae['pitch']} | Roll: {mae['roll']} | Total: {mae['total']}")
    print(f"  RMSE -> Yaw: {rmse['yaw']} | Pitch: {rmse['pitch']} | Roll: {rmse['roll']} | Total: {rmse['total']}")
    print(f"  R² -> Yaw: {r2['yaw']} | Pitch: {r2['pitch']} | Roll: {r2['roll']} | Total: {r2['total']}")

print_results("Test RAW:", mae_r, rmse_r, r2_r)
print_results("Test MASK:", mae_m, rmse_m, r2_m)
print_results("Test GLASSES:", mae_g, rmse_g, r2_g)
print_results("Test BOTH:", mae_b, rmse_b, r2_b)


-> Test 1/4: RAW


Evaluating: 100%|██████████| 6/6 [00:32<00:00,  5.39s/it]



-> Test 2/4: MASK


Evaluating: 100%|██████████| 6/6 [00:30<00:00,  5.15s/it]



-> Test 3/4: GLASSES


Evaluating: 100%|██████████| 6/6 [00:30<00:00,  5.02s/it]



-> Test 4/4: BOTH


Evaluating: 100%|██████████| 6/6 [00:30<00:00,  5.05s/it]


Test RAW:
  MAE -> Yaw: 0.12259089201688766 | Pitch: 0.1570242941379547 | Roll: 0.13681243360042572 | Total: 0.1388092041015625
  RMSE -> Yaw: 0.15527768433094025 | Pitch: 0.20764848589897156 | Roll: 0.1565522700548172 | Total: 0.17315948009490967
  R² -> Yaw: -0.22106695175170898 | Pitch: -46.26097106933594 | Roll: -1.1053576469421387 | Total: -15.862464904785156

Test MASK:
  MAE -> Yaw: 0.3934604227542877 | Pitch: 0.15674462914466858 | Roll: 0.08211459219455719 | Total: 0.21077321469783783
  RMSE -> Yaw: 0.4482227563858032 | Pitch: 0.20482006669044495 | Roll: 0.10380639135837555 | Total: 0.2522830665111542
  R² -> Yaw: -9.174407958984375 | Pitch: -44.982234954833984 | Roll: 0.07433187961578369 | Total: -18.027437210083008

Test GLASSES:
  MAE -> Yaw: 0.1747351437807083 | Pitch: 0.14715459942817688 | Roll: 0.1406664103269577 | Total: 0.15418539941310883
  RMSE -> Yaw: 0.20160287618637085 | Pitch: 0.1872008889913559 | Roll: 0.16447339951992035 | Total: 0.1844257265329361
  R² -> Yaw:

# Fine-Tunning

## Total Fine-Tunning

In [5]:
criterion = nn.L1Loss()

optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=True)

epochs = 5
best_val_loss = float('inf')

C:\Users\enric\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [6]:
train_losses = []
val_losses = []
def train_model(epochs=epochs, model=model, name = "base", train_loader=train_loader, val_loader=val_loader, criterion=criterion, optimizer=optimizer, scheduler=scheduler, device=device):
    global best_val_loss
    for epoch in range(epochs):
        start_time = time.time()

        # Training
        model.train()
        running_train_loss = 0.0
    
        for imgs, poses in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            imgs, poses = imgs.to(device), poses.to(device)
            
            optimizer.zero_grad()
            
            predictions = model(imgs)
            
            if isinstance(predictions, (tuple, list)):
                pred_mat = predictions[0]
                pred_angles = compute_euler_angles_from_rotation_matrices(pred_mat, use_gpu=True)
                pred_angles = pred_angles.to(device)
            else:
                pred_angles = predictions
                
            loss = criterion(pred_angles, poses)
            
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item() * imgs.size(0)
            
        epoch_train_loss = running_train_loss / len(train_dataset)
        train_losses.append(epoch_train_loss)
        
        # Validation
        model.eval()
        running_val_loss = 0.0
        
        with torch.no_grad():
            for imgs, poses in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
                imgs, poses = imgs.to(device), poses.to(device)
                
                predictions = model(imgs)
                
                if isinstance(predictions, (tuple, list)):
                    pred_mat = predictions[0]
                    pred_angles = compute_euler_angles_from_rotation_matrices(pred_mat, use_gpu=True)
                    pred_angles = pred_angles.to(device)
                else:
                    pred_angles = predictions
                    
                loss = criterion(pred_angles, poses)
                running_val_loss += loss.item() * imgs.size(0)
                
        epoch_val_loss = running_val_loss / len(val_dataset)
        val_losses.append(epoch_val_loss)
        
        scheduler.step(epoch_val_loss)
        
        epoch_time = time.time() - start_time
        
        print(f"\nEpoch {epoch+1}/{epochs} completed in {epoch_time:.0f}s")
        print(f"Train Loss (MAE radians): {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
        
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            save_path = "./weights/TokenHPE_Finetuned_" + name + ".pth"
            torch.save(model.state_dict(), save_path)
            print("Best model updated")
        else:
            print("\n")

In [7]:
train_model(name="Complete", epochs=epochs, model=model, optimizer=optimizer, scheduler=scheduler)

Epoch 1/5 [Val]: 100%|██████████| 3/3 [01:33<00:00, 31.13s/it]



Epoch 1/5 completed in 210s
Train Loss (MAE radians): 0.0793 | Val Loss: 0.1499
Best model updated


Epoch 2/5 [Val]: 100%|██████████| 3/3 [00:57<00:00, 19.07s/it]



Epoch 2/5 completed in 155s
Train Loss (MAE radians): 0.0415 | Val Loss: 0.1095
Best model updated


Epoch 3/5 [Val]: 100%|██████████| 3/3 [00:38<00:00, 12.86s/it]



Epoch 3/5 completed in 139s
Train Loss (MAE radians): 0.0355 | Val Loss: 0.1270




Epoch 4/5 [Val]: 100%|██████████| 3/3 [00:38<00:00, 12.76s/it]



Epoch 4/5 completed in 134s
Train Loss (MAE radians): 0.0263 | Val Loss: 0.1117




Epoch 5/5 [Val]:   0%|          | 0/3 [00:33<?, ?it/s]


KeyboardInterrupt: 

In [16]:
sys.path.append("./models")

from token_hpe import TokenHPE

model = TokenHPE(depth=3)
checkpoint_path = "./weights/TokenHPE_Finetuned_Complete.pth"
checkpoint = torch.load(checkpoint_path, map_location="cpu")

state_dict = checkpoint.get("model_state_dict", checkpoint)
model.load_state_dict(state_dict, strict=True)

model = model.to(device)
model.eval()

print("\n-> Test 1/4: RAW")
mae_r, rmse_r, r2_r = utils.evaluate_metrics(model, raw_loader, device)

print("\n-> Test 2/4: MASK")
mae_m, rmse_m, r2_m = utils.evaluate_metrics(model, mask_loader, device)

print("\n-> Test 3/4: GLASSES")
mae_g, rmse_g, r2_g = utils.evaluate_metrics(model, glasses_loader, device)

print("\n-> Test 4/4: BOTH")
mae_b, rmse_b, r2_b = utils.evaluate_metrics(model, both_loader, device)

def print_results(test_name, mae, rmse, r2):
    print(f"\n{test_name}")
    print(f"  MAE -> Yaw: {mae['yaw']} | Pitch: {mae['pitch']} | Roll: {mae['roll']} | Total: {mae['total']}")
    print(f"  RMSE -> Yaw: {rmse['yaw']} | Pitch: {rmse['pitch']} | Roll: {rmse['roll']} | Total: {rmse['total']}")
    print(f"  R² -> Yaw: {r2['yaw']} | Pitch: {r2['pitch']} | Roll: {r2['roll']} | Total: {r2['total']}")

print_results("Test RAW:", mae_r, rmse_r, r2_r)
print_results("Test MASK:", mae_m, rmse_m, r2_m)
print_results("Test GLASSES:", mae_g, rmse_g, r2_g)
print_results("Test BOTH:", mae_b, rmse_b, r2_b)


==> Add Sine PositionEmbedding~


C:\Users\enric\AppData\Local\Temp\ipykernel_9028\2272385563.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location="cpu")



-> Test 1/4: RAW


Evaluating: 100%|██████████| 6/6 [01:43<00:00, 17.32s/it]



-> Test 2/4: MASK


...


-> Test 3/4: GLASSES


Evaluating: 100%|██████████| 6/6 [01:50<00:00, 18.50s/it]



-> Test 4/4: BOTH


Evaluating: 100%|██████████| 6/6 [02:08<00:00, 21.41s/it]


Test RAW:
  MAE -> Yaw: 0.1443709284067154 | Pitch: 0.0600767582654953 | Roll: 0.09175396710634232 | Total: 0.09873387962579727
  RMSE -> Yaw: 0.18031920492649078 | Pitch: 0.06603863090276718 | Roll: 0.1237848550081253 | Total: 0.12338089942932129
  R² -> Yaw: -0.6466652154922485 | Pitch: -3.780146598815918 | Roll: -0.3162614107131958 | Total: -1.5810242891311646

Test MASK:
  MAE -> Yaw: 0.16326932609081268 | Pitch: 0.04877862334251404 | Roll: 0.08785069733858109 | Total: 0.09996622055768967
  RMSE -> Yaw: 0.20031437277793884 | Pitch: 0.055185288190841675 | Roll: 0.12056469917297363 | Total: 0.12535478174686432
  R² -> Yaw: -1.032102346420288 | Pitch: -2.338041305541992 | Roll: -0.24866950511932373 | Total: -1.2062710523605347

Test GLASSES:
  MAE -> Yaw: 0.13290010392665863 | Pitch: 0.04865995794534683 | Roll: 0.09325230121612549 | Total: 0.09160412102937698
  RMSE -> Yaw: 0.16485871374607086 | Pitch: 0.054508306086063385 | Roll: 0.12339601665735245 | Total: 0.11425434798002243
  R²

## LoRA

In [20]:
sys.path.append("./models")

from token_hpe import TokenHPE

model_lora = TokenHPE(depth=3)
checkpoint = torch.load("./weights/TokenHPEv1-ViTB-224_224-lyr3.tar", map_location="cpu")

state_dict = checkpoint["model_state_dict"]

model_lora.load_state_dict(state_dict, strict=True)

model_lora = model_lora.to(device)

==> Add Sine PositionEmbedding~


C:\Users\enric\AppData\Local\Temp\ipykernel_9028\2471208752.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("./weights/TokenHPEv1-ViTB-224_224-ly

In [21]:
for name, m in model_lora.feature_extractor.named_modules():
    if 'qkv' in name or 'attn' in name or 'proj' in name:
        print(name)

patch_embed.proj
blocks.0.attn
blocks.0.attn.qkv
blocks.0.attn.attn_drop
blocks.0.attn.proj
blocks.0.attn.proj_drop
blocks.1.attn
blocks.1.attn.qkv
blocks.1.attn.attn_drop
blocks.1.attn.proj
blocks.1.attn.proj_drop
blocks.2.attn
blocks.2.attn.qkv
blocks.2.attn.attn_drop
blocks.2.attn.proj
blocks.2.attn.proj_drop
blocks.3.attn
blocks.3.attn.qkv
blocks.3.attn.attn_drop
blocks.3.attn.proj
blocks.3.attn.proj_drop
blocks.4.attn
blocks.4.attn.qkv
blocks.4.attn.attn_drop
blocks.4.attn.proj
blocks.4.attn.proj_drop
blocks.5.attn
blocks.5.attn.qkv
blocks.5.attn.attn_drop
blocks.5.attn.proj
blocks.5.attn.proj_drop
blocks.6.attn
blocks.6.attn.qkv
blocks.6.attn.attn_drop
blocks.6.attn.proj
blocks.6.attn.proj_drop
blocks.7.attn
blocks.7.attn.qkv
blocks.7.attn.attn_drop
blocks.7.attn.proj
blocks.7.attn.proj_drop
blocks.8.attn
blocks.8.attn.qkv
blocks.8.attn.attn_drop
blocks.8.attn.proj
blocks.8.attn.proj_drop
blocks.9.attn
blocks.9.attn.qkv
blocks.9.attn.attn_drop
blocks.9.attn.proj
blocks.9.attn.pro

In [22]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["qkv"], # Targets the self-attention projections in the ViT blocks
    lora_dropout=0.1,
    bias="none"
)

model_lora.feature_extractor = get_peft_model(model_lora.feature_extractor, config)
model_lora = model_lora.to(device)

In [23]:
# We completely freeze all the parameters of the base model
for param in model_lora.parameters():
    param.requires_grad = False

# We only unfreeze the parameters that contain "lora" and the downstream task modules
for name, param in model_lora.named_parameters():
    if "lora" in name or "Ori_blocks" in name or "mlp_head" in name:
        param.requires_grad = True

# Here we can see how many parameters will actually be trained
trainable_params = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_lora.parameters())
print(f"Trainable Parameters: {trainable_params:,} out of {total_params:,} "
      f"({trainable_params/total_params:.2%})")

Trainable Parameters: 1,231,636 out of 87,034,906 (1.42%)


In [24]:
criterion = nn.L1Loss()

optimizer_lora = optim.AdamW(
    filter(lambda p: p.requires_grad, model_lora.parameters()),
    lr=5e-5,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_lora, mode='min', patience=3, factor=0.5, verbose=True
)

best_val_loss = float('inf')
epochs = 10
best_val_loss = float('inf')
train_model(epochs=epochs, model=model_lora, optimizer=optimizer_lora, scheduler=scheduler, name="LoRA")

Epoch 1/10 [Val]: 100%|██████████| 3/3 [00:38<00:00, 12.89s/it]



Epoch 1/10 completed in 210s
Train Loss (MAE radians): 0.0886 | Val Loss: 0.1291
Best model updated


Epoch 2/10 [Val]: 100%|██████████| 3/3 [00:35<00:00, 11.87s/it]



Epoch 2/10 completed in 164s
Train Loss (MAE radians): 0.0657 | Val Loss: 0.1558




Epoch 3/10 [Val]: 100%|██████████| 3/3 [00:39<00:00, 13.14s/it]


...


Epoch 8/10 completed in 187s
Train Loss (MAE radians): 0.0358 | Val Loss: 0.1290
Best model updated


Epoch 9/10 [Val]: 100%|██████████| 3/3 [00:46<00:00, 15.38s/it]



Epoch 9/10 completed in 239s
Train Loss (MAE radians): 0.0348 | Val Loss: 0.1187
Best model updated


Epoch 10/10 [Val]: 100%|██████████| 3/3 [00:43<00:00, 14.59s/it]


Epoch 10/10 completed in 242s
Train Loss (MAE radians): 0.0318 | Val Loss: 0.1274




In [25]:
model_lora.eval()

print("\n-> Test 1/4: RAW")
mae_r, rmse_r, r2_r = utils.evaluate_metrics(model_lora, raw_loader, device)

print("\n-> Test 2/4: MASK")
mae_m, rmse_m, r2_m = utils.evaluate_metrics(model_lora, mask_loader, device)

print("\n-> Test 3/4: GLASSES")
mae_g, rmse_g, r2_g = utils.evaluate_metrics(model_lora, glasses_loader, device)

print("\n-> Test 4/4: BOTH")
mae_b, rmse_b, r2_b = utils.evaluate_metrics(model_lora, both_loader, device)

def print_results(test_name, mae, rmse, r2):
    print(f"\n{test_name}")
    print(f"  MAE -> Yaw: {mae['yaw']} | Pitch: {mae['pitch']} | Roll: {mae['roll']} | Total: {mae['total']}")
    print(f"  RMSE -> Yaw: {rmse['yaw']} | Pitch: {rmse['pitch']} | Roll: {rmse['roll']} | Total: {rmse['total']}")
    print(f"  R² -> Yaw: {r2['yaw']} | Pitch: {r2['pitch']} | Roll: {r2['roll']} | Total: {r2['total']}")

print_results("Test RAW:", mae_r, rmse_r, r2_r)
print_results("Test MASK:", mae_m, rmse_m, r2_m)
print_results("Test GLASSES:", mae_g, rmse_g, r2_g)
print_results("Test BOTH:", mae_b, rmse_b, r2_b)


-> Test 1/4: RAW


Evaluating: 100%|██████████| 6/6 [00:54<00:00,  9.11s/it]



-> Test 2/4: MASK


Evaluating: 100%|██████████| 6/6 [00:53<00:00,  8.89s/it]



-> Test 3/4: GLASSES


Evaluating: 100%|██████████| 6/6 [00:54<00:00,  9.09s/it]



-> Test 4/4: BOTH


Evaluating: 100%|██████████| 6/6 [00:55<00:00,  9.22s/it]


Test RAW:
  MAE -> Yaw: 0.14021733403205872 | Pitch: 0.07690264284610748 | Roll: 0.07584682106971741 | Total: 0.09765559434890747
  RMSE -> Yaw: 0.1769261360168457 | Pitch: 0.08967607468366623 | Roll: 0.09845694154500961 | Total: 0.12168639153242111
  R² -> Yaw: -0.5852776765823364 | Pitch: -7.814511299133301 | Roll: 0.16727834939956665 | Total: -2.7441701889038086

Test MASK:
  MAE -> Yaw: 0.18028828501701355 | Pitch: 0.0917685404419899 | Roll: 0.09340845048427582 | Total: 0.12182176113128662
  RMSE -> Yaw: 0.2214042991399765 | Pitch: 0.10212746262550354 | Roll: 0.11273400485515594 | Total: 0.14542192220687866
  R² -> Yaw: -1.4825241565704346 | Pitch: -10.432210922241211 | Roll: -0.09173452854156494 | Total: -4.002156734466553

Test GLASSES:
  MAE -> Yaw: 0.12621517479419708 | Pitch: 0.06950205564498901 | Roll: 0.08978792279958725 | Total: 0.09516838192939758
  RMSE -> Yaw: 0.1580347716808319 | Pitch: 0.08018095791339874 | Roll: 0.11482927203178406 | Total: 0.11768165975809097
  R² -

# TODO list

## Probar con oclusion sencilla

## Buscar como hacer un test con imagenes reales

## Cambiar métricas a RMSE y R² (o MAPE con comprobacion de que no divida entre 0)



In [2]:
from datasets import load_dataset

# Carregar el dataset (descarrega automàticament les imatges i anotacions)
dataset = load_dataset("ETHZurich/biwi_kinect_head_pose", trust_remote_code=True)

# Exemple d'estructura d'una mostra:
sample = dataset['train'][0]
print(sample.keys())
# Retorna dict_keys(['sequence_number', 'subject_id', 'rgb', 'rgb_cal', 'depth'])

FileNotFoundError: https://data.vision.ee.ethz.ch/cvl/gfanelli/kinect_head_pose_db.tgz

In [8]:
import scipy.io
import torch

mat_data = scipy.io.loadmat('p02_image_04.mat')
vector_6d = mat_data['HP_camera'][0]  

print("keys:", mat_data.keys())

angle_1 = vector_6d[3]
angle_2 = vector_6d[4]
angle_3 = vector_6d[5]

print(f"Angle 1: {angle_1}, Angle 2: {angle_2}, Angle 3: {angle_3}")

radians = torch.tensor([angle_1, angle_2, angle_3]) * (torch.pi / 180)
print(radians)

keys: dict_keys(['__header__', '__version__', '__globals__', 'HP_camera'])
Angle 1: 7.0333972381660965, Angle 2: 23.097407019507475, Angle 3: 12.855895204825982
tensor([0.1228, 0.4031, 0.2244], dtype=torch.float64)


In [9]:
import scipy.io as sio


hp = mat_data["HP_camera"]
print(hp.shape)
print(hp[0])


(1, 6)
[  -7.91547312 -172.01305958  638.96059032    7.03339724   23.09740702
   12.8558952 ]
